# Session 02 — Causal Shifting, Loss Masks, and Hidden Gradient Paths

This session asks a deceptively simple question: **which prediction is being graded?** Stop at each checkpoint, make a qualitative prediction, then compare with the adjacent runnable reference solution.

In [ ]:
import math
import torch
import torch.nn.functional as F

torch.manual_seed(7)
print('PyTorch:', torch.__version__)

## 1. One sequence contains several next-token examples

For `<BOS> I love dogs <EOS>`, decide which target belongs to the logits emitted at each input position. Remember: the state at position $t$ has seen $x_{\le t}$.

In [ ]:
tokens = ['<BOS>', 'I', 'love', 'dogs', '<EOS>']
input_ids = torch.tensor([[0, 1, 2, 3, 4]])  # [B=1, T=5]
toy_logits = torch.randn(1, 5, 5)             # [B, T, V]

def shift_for_next_token(logits, ids):
    # Replace each ... after making your prediction.
    shifted_logits = ...
    shifted_labels = ...
    return shifted_logits, shifted_labels

### Reference solution — align each position with its next token

In [ ]:
def reference_shift_for_next_token(logits, ids):
    return logits[:, :-1, :], ids[:, 1:]

shift_logits, shift_labels = reference_shift_for_next_token(toy_logits, input_ids)
print('shifted logits shape:', tuple(shift_logits.shape))
print('shifted labels shape:', tuple(shift_labels.shape))
print('targets:', [tokens[i] for i in shift_labels[0].tolist()])
assert shift_logits.shape == (1, 4, 5)
assert shift_labels.tolist() == [[1, 2, 3, 4]]

**Why this works:** logits at position 0 are produced from `<BOS>` and are graded against `I`; logits at position 3 are produced from the prefix ending in `dogs` and are graded against `<EOS>`. Position 4 has no later target inside this sequence, so its logits are dropped. The shift aligns a prediction with its target; the causal attention mask separately controls what the prediction was allowed to see.

## 2. A low loss can certify the wrong task

The next cell constructs a fake model that confidently copies the token at its current position. Predict which loss will be small: comparison with current tokens, or comparison with next tokens?

In [ ]:
V = 5
copy_logits = torch.full((1, 5, V), -10.0)
copy_logits.scatter_(2, input_ids.unsqueeze(-1), 10.0)

unshifted_loss = F.cross_entropy(copy_logits.reshape(-1, V), input_ids.reshape(-1))
next_token_loss = F.cross_entropy(
    copy_logits[:, :-1, :].reshape(-1, V),
    input_ids[:, 1:].reshape(-1),
)
print('unshifted copy loss: ', float(unshifted_loss))
print('true next-token loss:', float(next_token_loss))
assert unshifted_loss < 1e-6
assert next_token_loss > 10

**Interpretation:** near-zero unshifted loss does not show next-token ability. It shows that a current-token copying objective was solved. This is target leakage through objective alignment, not necessarily future-token leakage through attention; those are distinct bugs.

## 3. Loss masks grade only selected targets

Imagine that the first three shifted targets belong to a prompt and only `dogs` plus `<EOS>` should be supervised. Implement a mean that divides by valid targets only.

In [ ]:
labels_with_ignore = torch.tensor([[-100, -100, 3, 4]])

def masked_mean_cross_entropy(logits, labels):
    # Return a scalar mean over labels that are not -100.
    return ...

### Reference solution — valid-target denominator

In [ ]:
def reference_masked_mean_cross_entropy(logits, labels):
    V = logits.shape[-1]
    return F.cross_entropy(
        logits.reshape(-1, V),
        labels.reshape(-1),
        ignore_index=-100,
        reduction='mean',
    )

masked_loss = reference_masked_mean_cross_entropy(shift_logits, labels_with_ignore)
per_position = F.cross_entropy(
    shift_logits.reshape(-1, 5),
    labels_with_ignore.reshape(-1),
    ignore_index=-100,
    reduction='none',
).reshape_as(labels_with_ignore)
valid = labels_with_ignore.ne(-100)
manual = per_position[valid].sum() / valid.sum()
print('valid targets:', int(valid.sum()))
print('library mean:', float(masked_loss))
print('manual mean: ', float(manual))
assert torch.allclose(masked_loss, manual)

**Why this works:** ignored locations contribute neither a loss term nor a count in the denominator. Dividing by padded capacity would make otherwise identical examples receive different gradient scale merely because their storage shapes differ. An attention mask answers a different question: whether a state may read a position.

## 4. A masked prompt can still receive gradient influence

The following tiny causal attention graph assigns loss only to the prediction of `Paris`. Before running it, predict whether the input representations of earlier prompt tokens will have zero or nonzero gradient.

In [ ]:
torch.manual_seed(11)
names = ['<BOS>', 'capital', 'of', 'France', '<ASSISTANT>', 'Paris', '<EOS>']
ids = torch.arange(len(names)).unsqueeze(0)
T, D, V = len(names), 8, len(names)

embedding = torch.nn.Embedding(V, D)
q_proj = torch.nn.Linear(D, D, bias=False)
k_proj = torch.nn.Linear(D, D, bias=False)
v_proj = torch.nn.Linear(D, D, bias=False)
lm_head = torch.nn.Linear(D, V, bias=False)

x = embedding(ids)
x.retain_grad()
q, k, v = q_proj(x), k_proj(x), v_proj(x)
scores = q @ k.transpose(-1, -2) / math.sqrt(D)
causal_mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
attention = torch.softmax(scores.masked_fill(causal_mask, float('-inf')), dim=-1)
hidden = attention @ v
logits = lm_head(hidden)

# Position 4 (<ASSISTANT>) predicts position 5 (Paris); all other targets are ignored.
shifted_labels = torch.full((1, T - 1), -100, dtype=torch.long)
shifted_labels[0, 4] = ids[0, 5]
loss = F.cross_entropy(logits[:, :-1, :].reshape(-1, V), shifted_labels.reshape(-1))
loss.backward()

for position, name in enumerate(names[:5]):
    print(f'{name:12s} input-state gradient norm: {x.grad[0, position].norm().item():.6f}')
assert all(x.grad[0, position].norm() > 0 for position in range(5))

**Interpretation:** the answer loss depends on an answer query that reads the prompt's keys and values, so gradient flows into the prompt input representations even without local prompt losses. This cell observes input-side states; it does not claim that every final-layer prompt activation must have a gradient. Loss masking removes selected objective terms—it does not detach contextual computation.

## Final synthesis question

A prompt token is readable by answer positions but has label `-100`. State separately what this implies for (1) attention visibility, (2) direct loss contribution, and (3) possible gradient flow.

<details>
<summary><strong>Reference synthesis</strong></summary>

The token remains visible wherever the causal and padding/document attention rules permit it. Its `-100` target contributes no direct cross-entropy term and is absent from the valid-target mean denominator. Nevertheless, supervised answer predictions can depend on it through attention, so gradients can flow through its key/value path into earlier prompt states, input embeddings, and shared model parameters. Visibility, grading, and differentiability are related through the computation graph but are not the same control.
</details>